# Maithili SUMMARY — ZERO-SHOT — `llama31`

**Model:** `meta-llama/Llama-3.1-8B-Instruct` · **Test split:** `mai_test.csv` (201 rows) · **Shots:** 0

Your summary prompt is reproduced verbatim with only the exemplar block removed
(prompt sha1 of the string values = `4c51a6e2b34cbd0f`). Affect-preservation clauses retained.

| setting | value | matches fine-tuned IndicBART |
|---|---|---|
| eval set | 201-row test split | yes |
| decoding | beams=4, len_pen=1.0, no_repeat=3 | yes |
| `max_new_tokens` | 200 | yes |
| precision | bfloat16 (no bitsandbytes) | — |
| post-processing | cleanup only, no length cap | yes (none) |
| empty output | scored 0, never dropped | yes |

**Before running:** Kernel → Restart, then Run All from the top. Cell 0 must run
before anything imports `torch`.


In [1]:
# ============================================================
# CELL 0 — GPU selection + environment. RUN FIRST, after Kernel -> Restart.
# ------------------------------------------------------------
# On a SHARED multi-GPU machine two things must happen BEFORE `import torch`:
#   1. choose ONE GPU (the one with most free memory) via CUDA_VISIBLE_DEVICES.
#      Setting it after torch is imported has no effect.
#   2. keep the model on that single GPU. device_map="auto" otherwise spreads
#      layers across every visible card — including ones other users have filled.
# ============================================================
import os, sys, shutil, subprocess

if "torch" in sys.modules:
    raise SystemExit("torch is already imported. Kernel -> Restart Kernel, then run this cell first.")

# ---- 1. find the GPU with the most free memory (nvidia-smi, no torch yet) ----
_rows = subprocess.run(
    ["nvidia-smi", "--query-gpu=index,memory.free,memory.total,utilization.gpu",
     "--format=csv,noheader,nounits"],
    capture_output=True, text=True).stdout.strip().split("\n")
_gpus = []
for r in _rows:
    i, free, tot, util = [x.strip() for x in r.split(",")]
    _gpus.append((int(i), int(free), int(tot), int(util)))
    print(f"  GPU {i}: {int(free)/1024:5.1f} GB free of {int(tot)/1024:.1f} GB | util {util}%")

GPU_ID, _free_mb, _tot_mb, _ = max(_gpus, key=lambda g: g[1])
os.environ["CUDA_VISIBLE_DEVICES"] = str(GPU_ID)           # BEFORE torch import
os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"
print(f"\n-> using physical GPU {GPU_ID} ({_free_mb/1024:.1f} GB free); "
      f"it appears as cuda:0 inside this kernel")

# ---- 2. environment (no compiler on some nodes -> bf16, no Triton) ----
for _k, _v in {"TORCHINDUCTOR_DISABLE":"1", "TORCHDYNAMO_DISABLE":"1",
               "TORCH_COMPILE_DISABLE":"1", "BITSANDBYTES_NOWELCOME":"1",
               "TOKENIZERS_PARALLELISM":"false", "HF_HUB_DISABLE_XET":"1",
               "HF_HUB_DOWNLOAD_TIMEOUT":"60"}.items():
    os.environ[_k] = _v
for _v in ("LIBRARY_PATH", "LD_LIBRARY_PATH"):
    _cur = os.environ.get(_v, "")
    if "/lib64" not in _cur.split(":"):
        os.environ[_v] = f"/lib64:{_cur}" if _cur else "/lib64"
shutil.rmtree(os.path.expanduser("~/.triton/cache"), ignore_errors=True)

import torch
torch._dynamo.config.suppress_errors = True
assert torch.cuda.device_count() == 1, "more than one GPU still visible — restart the kernel"
_free, _total = torch.cuda.mem_get_info(0)
VRAM_GB = _free / (1024 ** 3)
print(f"torch {torch.__version__} | cuda:0 = {torch.cuda.get_device_name(0)} | "
      f"{VRAM_GB:.1f} GB free")


  GPU 0:  23.2 GB free of 24.0 GB | util 0%
  GPU 1:  23.7 GB free of 24.0 GB | util 0%

-> using physical GPU 1 (23.7 GB free); it appears as cuda:0 inside this kernel


torch 2.14.0+cu126 | cuda:0 = NVIDIA RTX A5000 | 23.5 GB free


In [2]:
import os
os.environ["CUDA_VISIBLE_DEVICES"] = "0,1"

In [3]:
# ============================================================
# CELL 1 — configuration (nothing to edit)
# ============================================================
MODEL_KEY  = "llama31"
MODEL_NAME = "meta-llama/Llama-3.1-8B-Instruct"
ARCH       = "causal"         # causal | seq2seq
FMT        = "chat_sys"   # chat-template wrapper — the ONLY per-model difference
GATED      = True

LANG, TASK = "mai", "summary"
TEST_CSV, EXPECT_N = "mai_test.csv", 201

SHOTS            = 0
NUM_BEAMS        = 4
LENGTH_PENALTY   = 1.0
NO_REPEAT_NGRAM  = 3
DO_SAMPLE        = False
MAX_NEW_TOKENS   = 200          # same as fine-tuned IndicBART summary
MAX_INPUT_TOKENS = 1024
USE_4BIT         = False        # bf16: bitsandbytes needs a compiler this node lacks
BATCH_SIZE       = 1            # beams=4 x 1224-token context; raise only if memory allows
SEED             = 42
FRESH_START      = True

NEED_GB = 18
print(f"RUN {MODEL_KEY} | {LANG}/{TASK} | beams={NUM_BEAMS} max_new={MAX_NEW_TOKENS} "
      f"bf16 | shots={SHOTS}")
if VRAM_GB < NEED_GB:
    raise SystemExit(
        f"{MODEL_KEY} needs ~{NEED_GB} GB in bf16, but the freest GPU has only "
        f"{VRAM_GB:.1f} GB.\n"
        f"  1. check your OWN stale processes:  !nvidia-smi\n"
        f"     then Kernel -> Shut Down All Kernels (or kill that PID)\n"
        f"  2. or run a smaller model first: indicbart_pre (3 GB), sarvam1 (6 GB)")

RUN_TAG        = f"{MODEL_KEY}_{LANG}_{TASK}_zeroshot"
CHECKPOINT_CSV = f"ckpt_{RUN_TAG}.csv"
FINAL_CSV      = f"gen_{RUN_TAG}.csv"
PERITEM_CSV    = f"per_item_{RUN_TAG}.csv"
METRICS_JSON   = f"metrics_{RUN_TAG}.json"
MANIFEST_JSON  = f"manifest_{RUN_TAG}.json"


RUN llama31 | mai/summary | beams=4 max_new=200 bf16 | shots=0


In [4]:
# ============================================================
# CELL 2 — imports, seeding, GPU guard, HF auth (gated only)
# ============================================================
import re, gc, json, time, glob, random, hashlib, unicodedata
import numpy as np, pandas as pd

random.seed(SEED); np.random.seed(SEED)
torch.manual_seed(SEED); torch.cuda.manual_seed_all(SEED)
assert torch.cuda.is_available(), "No GPU — a CPU fallback would take days."
print("device:", torch.cuda.get_device_name(0))

if GATED:
    from huggingface_hub import login, whoami, hf_hub_download
    tok = os.environ.get("HF_TOKEN")
    if not tok:
        import getpass
        tok = getpass.getpass("HF token (hidden): ").strip()
        os.environ["HF_TOKEN"] = tok
    login(tok, add_to_git_credential=False)
    print("logged in as:", whoami()["name"])
    try:
        hf_hub_download(MODEL_NAME, "config.json"); print("access OK")
    except Exception as e:
        raise SystemExit(f"No access to {MODEL_NAME}: {str(e)[:150]}\n"
                         f"Check https://huggingface.co/settings/gated-repos")


device: NVIDIA RTX A5000


Note: Environment variable`HF_TOKEN` is set and is the current active token independently from the token you've just configured.


logged in as: prakashdeepak132


access OK


In [5]:
# ============================================================
# CELL 3 — load the TEST SPLIT (self-healing)
# ============================================================
TEXT_COL, REF_COL, GEN_COL = "article_text", "summary", "generated"

cands = [TEST_CSV] + sorted(set(glob.glob("*test*.csv")))
path = next((p for p in cands if os.path.exists(p)), None)
if path is None:
    raise SystemExit(f"No test CSV in {os.getcwd()}. Expected {TEST_CSV}.")
if path != TEST_CSV:
    print(f"NOTE: using '{path}' (did not find {TEST_CSV})")

df = pd.read_csv(path, dtype=str, keep_default_na=False)
alias = {"article_text":["article_text","text","article","body","content"],
         "summary":["summary","gold_summary","reference_summary"],
         "headline":["headline","article_headline","title"]}
low = {c.strip().lower(): c for c in df.columns}
for want, alts in alias.items():
    if want not in df.columns:
        hit = next((low[a] for a in alts if a in low), None)
        if hit: df = df.rename(columns={hit: want}); print(f"  renamed {hit} -> {want}")
for c in (TEXT_COL, REF_COL):
    assert c in df.columns, f"missing '{c}'. Have: {list(df.columns)}"
    df[c] = df[c].astype(str).str.strip()
if "row_key" not in df.columns:
    df["row_key"] = df[TEXT_COL].map(lambda t: hashlib.sha1(t.encode()).hexdigest()[:16])
if not df["row_key"].is_unique:
    df["row_key"] = [f"{k}_{i}" for i, k in enumerate(df["row_key"])]

if len(df) != EXPECT_N:
    print(f"\n!!! {len(df)} rows, expected {EXPECT_N} (Table 6). Scores will NOT be "
          f"comparable to fine-tuned IndicBART. Continuing; true n is recorded.\n")
    EXPECT_N = len(df)
else:
    print(f"OK: {len(df)} rows matches Table 6")

df[GEN_COL] = ""
w = df[REF_COL].str.split().str.len()
print(f"gold summary words: mean={w.mean():.1f} median={w.median():.0f} "
      f"min={w.min()} max={w.max()}")


OK: 201 rows matches Table 6
gold summary words: mean=79.5 median=79 min=43 max=114


In [6]:
# ============================================================
# CELL 4 — article cleaning
# ============================================================
_ZW = dict.fromkeys(map(ord, "\u200b\u200c\u200d\u200e\u200f\ufeff"), None)
def clean_article(t):
    s = "" if t is None else str(t)
    if s.strip().lower() in {"nan","none","null"}: return ""
    s = unicodedata.normalize("NFC", s).translate(_ZW)
    s = "".join(ch for ch in s if unicodedata.category(ch)[0] != "C" or ch in "\t\n ")
    return re.sub(r"\s+", " ", s).strip()
df["_clean"] = df[TEXT_COL].map(clean_article)
print("empty articles:", int((df["_clean"] == "").sum()))


empty articles: 3


In [7]:
# ============================================================
# CELL 5 — PROMPT (your Maithili summary prompt, verbatim; exemplar REMOVED)
# ============================================================
SYSTEM_PROMPT = 'अहाँ एकटा मैथिली समाचार सारांश लेखक छी। अहाँ हमेशा शुद्ध मैथिली मे लिखब, हिंदी मे कहियो नहि लिखब। अहाँ लेख केर मूल भाव आ भावना केँ जेना केर तेना राखब।'

def _instruction_block():
    return "नीचाँ देल गेल मैथिली समाचार लेख केर मैथिली भाषा मे 80 सँ 100 शब्द केर सारांश लिखू। सारांश खाली शुद्ध मैथिली मे होअय केर चाही — हिंदी मे नहि लिखू। मैथिली केर शब्द इस्तेमाल करू जेना: अछि, छथि, छल, गेल, कयलनि, रहल, होअय, चाही, अहाँ, हमरा, ओकर, एकर, सँ, केँ, मे।\n\nआओर जरूरी बात:\n- सारांश 80 सँ 100 शब्द केर बीच मे होअय। 100 शब्द सँ बेसी नहि लिखू।\n- लेख केर भाव (खुशी, दुख, चिंता, गुस्सा, आशा, निराशा, उत्साह) आ स्वर केँ जेना केर तेना राखू। लेख केर भाव केँ हल्का वा साधारण नहि करू।\n- लेख मे जे नाम, जगह, संख्या आ तारीख अछि, ओकरा सारांश मे शामिल करू।\n- अपन तरफ सँ कोनो नव जानकारी वा राय नहि जोड़ू। जे लेख मे अछि, खाली ओकरे आधार पर लिखू।\n- खाली एक बेर मे एक संग सारांश लिखू। कोनो शीर्षक, 'सारांश:' जेना शब्द, बुलेट वा नंबर नहि जोड़ू।"

def build_user_prompt(cleaned):
    """ZERO-SHOT: instruction + article. The original's exemplar block is absent."""
    return (_instruction_block()
            + f"\n\nलेख:\n{cleaned}\n\n"
            + 'मैथिली सारांश (80-100 शब्द, खाली मैथिली मे):')

def format_input(cleaned):
    u = build_user_prompt(cleaned)
    if FMT == "airavata":        # <|system|>/<|user|>/<|assistant|>, add_bos=False as in yours
        return f"<|system|>\n{SYSTEM_PROMPT}\n<|user|>\n{u}\n<|assistant|>\n"
    if FMT == "plain":           # base LMs without a chat template
        return f"{SYSTEM_PROMPT}\n\n{u}"
    if FMT == "chat_sys":        # Llama-3.1
        return tokenizer.apply_chat_template(
            [{"role":"system","content":SYSTEM_PROMPT},{"role":"user","content":u}],
            tokenize=False, add_generation_prompt=True)
    if FMT == "chat_nosys":      # Gemma-2 has no system role
        return tokenizer.apply_chat_template(
            [{"role":"user","content":SYSTEM_PROMPT + "\n\n" + u}],
            tokenize=False, add_generation_prompt=True)
    if FMT == "none":            # IndicBART: no instruction, language tag only
        return f"<2hi> {cleaned} </s>"
    raise ValueError(FMT)

PROMPT_SHA1 = hashlib.sha1((SYSTEM_PROMPT + _instruction_block()).encode()).hexdigest()[:16]
print("prompt sha1 =", PROMPT_SHA1, "  (expected 4c51a6e2b34cbd0f)")
assert PROMPT_SHA1 == "4c51a6e2b34cbd0f", "prompt text drifted from the verified version"
print("SHOTS = 0 -> no exemplar")


prompt sha1 = 4c51a6e2b34cbd0f   (expected 4c51a6e2b34cbd0f)
SHOTS = 0 -> no exemplar


In [8]:
# ============================================================
# CELL 6 — load model (bf16, eager attention, no bitsandbytes)
# ============================================================
from transformers import AutoTokenizer, AutoModelForCausalLM, MBartForConditionalGeneration

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME, use_fast=True)
if ARCH == "seq2seq":
    model = MBartForConditionalGeneration.from_pretrained(
        MODEL_NAME, torch_dtype=torch.float32).cuda()
else:
    if tokenizer.pad_token is None:
        tokenizer.pad_token = tokenizer.eos_token
    tokenizer.padding_side = "left"
    model = AutoModelForCausalLM.from_pretrained(
        MODEL_NAME, device_map={"": 0}, torch_dtype=torch.bfloat16,
        low_cpu_mem_usage=True, attn_implementation="eager")
model.eval()
print(f"loaded {MODEL_NAME} | {torch.cuda.memory_allocated()/1e9:.1f} GB")


[transformers] `torch_dtype` is deprecated! Use `dtype` instead!


Loading weights:   0%|          | 0/291 [00:00<?, ?it/s]

loaded meta-llama/Llama-3.1-8B-Instruct | 16.1 GB


In [9]:
# ============================================================
# CELL 7 — post-processing (NO length cap) + generation
# ============================================================
_MARKERS = ("<|assistant|>","<|user|>","<|system|>","<|end|>","<|eot_id|>",
            "<|start_header_id|>","<|end_header_id|>","</s>","<s>","[/INST]",
            "<end_of_turn>","<start_of_turn>","<2hi>")

def postprocess(t):
    if not t: return ""
    s = str(t)
    for m in _MARKERS: s = s.replace(m, " ")
    s = re.sub(r"^\s*(मैथिली\s*|भोजपुरी\s*)?(सारांश|summary)\s*[:：\-]\s*", "", s, flags=re.I)
    return re.sub(r"\s+", " ", s).strip().strip(' "\u201c\u201d')

NTOK = {}   # generated-token count per row, to measure truncation at the cap

@torch.no_grad()
def generate_batch(cleans):
    enc = tokenizer([format_input(c) for c in cleans], return_tensors="pt",
                    padding=True, truncation=True, max_length=MAX_INPUT_TOKENS,
                    add_special_tokens=(ARCH != "seq2seq")).to(model.device)
    kw = dict(max_new_tokens=MAX_NEW_TOKENS, num_beams=NUM_BEAMS,
              length_penalty=LENGTH_PENALTY, no_repeat_ngram_size=NO_REPEAT_NGRAM,
              do_sample=DO_SAMPLE, early_stopping=True)
    if ARCH == "seq2seq":
        kw["decoder_start_token_id"] = tokenizer.convert_tokens_to_ids("<2hi>")
        out = model.generate(**enc, **kw)
        toks = out
    else:
        kw["pad_token_id"] = tokenizer.pad_token_id
        out = model.generate(**enc, **kw)
        toks = out[:, enc["input_ids"].shape[1]:]
    pad = tokenizer.pad_token_id
    lens = [int((row != pad).sum()) for row in toks]
    dec = tokenizer.batch_decode(toks, skip_special_tokens=True)
    return [postprocess(t) for t in dec], lens

t0 = time.time()
g, l = generate_batch(df["_clean"].head(2).tolist())
for i, (x, n) in enumerate(zip(g, l)):
    print(f"SMOKE {i}: [{n} tok, {len(x.split())} w] {x[:120]}")
est = (time.time() - t0) / 2 * len(df) / 60
print(f"est. ~{est:.0f} min for {len(df)} rows")


[transformers] Ignoring clean_up_tokenization_spaces=True for BPE tokenizer TokenizersBackend. The clean_up_tokenization post-processing step is designed for WordPiece tokenizers and is destructive for BPE (it strips spaces before punctuation). Set clean_up_tokenization_spaces=False to suppress this warning, or set clean_up_tokenization_spaces_for_bpe_even_though_it_will_corrupt_output=True to force cleanup anyway.


SMOKE 0: [65 tok, 19 w] अछी। अर्थार्जन के साथ-साथ हमरा सामाजिक, सांस्कृतिक, राजनीतिक आ शैक्षिक क्षेत्र मे भी प्रगति करय के प्रयास करय चाही।
SMOKE 1: [200 tok, 74 w] अर्थ: प्रवीन नारायन चौधारी के चंदन के कार्तून में निज भाषा, मिथली को छोड़कर अश्लिल और फूहरता से भरपूर बिहारी फिल्मों की 
est. ~30 min for 201 rows


In [10]:
# ============================================================
# CELL 8 — generation loop (fresh start, checkpointed, OOM-safe)
# ============================================================
if os.path.exists(CHECKPOINT_CSV):
    if FRESH_START:
        os.rename(CHECKPOINT_CSV, f"{CHECKPOINT_CSV}.bak.{int(time.time())}")
        print("old checkpoint moved aside")
    else:
        ck = pd.read_csv(CHECKPOINT_CSV, dtype=str, keep_default_na=False)
        m = dict(zip(ck["row_key"], ck[GEN_COL])); df[GEN_COL] = df["row_key"].map(lambda k: m.get(k, ""))

todo = df.index[df[GEN_COL].str.strip() == ""].tolist()
print(f"generating {len(todo)}/{len(df)}")
t0 = time.time()
for s in range(0, len(todo), BATCH_SIZE):
    idx = todo[s:s+BATCH_SIZE]
    batch = df.loc[idx, "_clean"].tolist()
    try:
        outs, lens = generate_batch(batch)
    except Exception as e:
        print(f"  [batch {s}: {type(e).__name__}] retrying singly")
        torch.cuda.empty_cache(); outs, lens = [], []
        for a in batch:
            try:  o, n = generate_batch([a]); outs.append(o[0]); lens.append(n[0])
            except Exception: outs.append(""); lens.append(0)
    for i, o, n in zip(idx, outs, lens):
        df.at[i, GEN_COL] = o; NTOK[df.at[i, "row_key"]] = n
    df[["row_key", GEN_COL]].to_csv(CHECKPOINT_CSV, index=False, encoding="utf-8-sig")
    if (s // BATCH_SIZE) % 10 == 0 or s + BATCH_SIZE >= len(todo):
        d = min(s + BATCH_SIZE, len(todo)); el = time.time() - t0
        print(f"  {d}/{len(todo)} | {el/60:.1f} min | ~{el/max(d,1)*(len(todo)-d)/60:.1f} min left")

print(f"done in {(time.time()-t0)/60:.1f} min | empty "
      f"{int((df[GEN_COL].str.strip()=='').sum())}/{len(df)}")
df.drop(columns=["_clean"]).to_csv(FINAL_CSV, index=False, encoding="utf-8-sig")
for i in df.index[:3]:
    print(f"\nGOLD: {df.at[i, REF_COL][:140]}\nGEN : {df.at[i, GEN_COL][:140]}")


generating 201/201


  1/201 | 0.1 min | ~22.3 min left


  11/201 | 1.5 min | ~26.0 min left


  21/201 | 2.6 min | ~22.4 min left


  31/201 | 3.9 min | ~21.5 min left


...

  161/201 | 18.8 min | ~4.7 min left


  171/201 | 20.0 min | ~3.5 min left


  181/201 | 21.2 min | ~2.3 min left


  191/201 | 22.5 min | ~1.2 min left


  201/201 | 23.7 min | ~0.0 min left
done in 23.7 min | empty 0/201

GOLD: १०० वर्षक बाद मिथिलाक भविष्य पर विचार करैत, आजुक युवा विश्व परिवेश संग डेग बढा रहल छथि। बाबा आदम वला कहावत मिथक बनि गेल अछि, आ आजुक युग मे अ
GEN : अछी। अर्थार्जन के साथ-साथ हमरा सामाजिक, सांस्कृतिक, राजनीतिक आ शैक्षिक क्षेत्र मे भी प्रगति करय के प्रयास करय चाही। आजुक युवा पीढ़ी के लेल अ

GOLD: प्रवीण नारायण चौधरी झा चन्दन केर कार्टून मैथिली सिनेमा सँ भोजपुरी सिनेमा दिस समाजक झुकाव बढ़बैत अछि। मिथिलावासी अपन मूल्य केँ अवमूल्यन करैत 
GEN : अर्थ: प्रवीन नारायन चौधारी जी के कार्तून मे मिथीली भासा का फिल्म छोड़कर अश्लिल और फूहड़ भोज पुरी फिलिम की ओर समाज का झूकाव बढ़ता जा रहा है। 

GOLD: डा. उपेन्द्र ठाकुरक "प्रारम्भिक राजतंत्रः विदेह" पुस्तकमे विदेह प्रदेश आ एकर जनता पर चर्चा कयल गेल अछि। विदेह प्राचीन आर्यभूमिक एक महत्वपूर्
GEN : जातिसभक नाम लिखल गेलैक, जिनमे सँ वैश्य, क्षत्रिय, शूद्र आ ब्राम्हण छथि। ५ वैश्यक समुदाय के बारे मे विशेष रूपसँ जानल जाएलैक जे क्रियाशील व्या


In [11]:
# ============================================================
# CELL 9 — STANDARDIZED EVALUATION  (no regeneration)
# ------------------------------------------------------------
# Loads existing generations — from memory if the kernel is alive, otherwise
# from gen_*.csv / ckpt_*.csv on disk. Nothing is regenerated.
#
# FIX: bert-score crashes on EMPTY strings with newer transformers
#   (BertTokenizer has no attribute build_inputs_with_special_tokens).
#   Empty outputs are now scored 0.0 on every metric WITHOUT being sent to
#   BERTScore. They are still counted in n, never dropped.
# ============================================================
import os, json, glob, hashlib
import numpy as np, pandas as pd

# ---- 1. recover config if the kernel was restarted ----
_g = globals()
LANG      = _g.get("LANG", "mai")
TASK      = _g.get("TASK", "summary")
MODEL_KEY = _g.get("MODEL_KEY", "llama31")
SEED      = _g.get("SEED", 42)
MAX_NEW_TOKENS = _g.get("MAX_NEW_TOKENS", 200)
REF_COL   = _g.get("REF_COL", "summary")
GEN_COL   = _g.get("GEN_COL", "generated")
MODEL_NAME = _g.get("MODEL_NAME", MODEL_KEY)
SHOTS     = _g.get("SHOTS", 0)
RUN_TAG   = _g.get("RUN_TAG", f"{MODEL_KEY}_{LANG}_{TASK}_zeroshot")
TEST_CSV  = _g.get("TEST_CSV", f"{LANG}_test.csv")
PERITEM_CSV  = f"per_item_{RUN_TAG}.csv"
METRICS_JSON = f"metrics_{RUN_TAG}.json"
NTOK = _g.get("NTOK", {})

# ---- 2. get generations WITHOUT regenerating ----
if "df" in _g and GEN_COL in df.columns and (df[GEN_COL].astype(str).str.strip() != "").any():
    ev = df.copy()
    print("using generations already in memory")
else:
    ref = pd.read_csv(TEST_CSV, dtype=str, keep_default_na=False)
    if "row_key" not in ref.columns:
        ref["row_key"] = ref["article_text"].astype(str).str.strip().map(
            lambda t: hashlib.sha1(t.encode()).hexdigest()[:16])
    src = next((p for p in (f"gen_{RUN_TAG}.csv", f"ckpt_{RUN_TAG}.csv") if os.path.exists(p)), None)
    if src is None:
        raise SystemExit(f"No saved generations for {RUN_TAG}. Looked for gen_/ckpt_ CSVs in {os.getcwd()}")
    g = pd.read_csv(src, dtype=str, keep_default_na=False)
    m = dict(zip(g["row_key"], g[GEN_COL]))
    ev = ref.copy()
    ev[GEN_COL] = ev["row_key"].map(lambda k: m.get(k, ""))
    print(f"loaded generations from {src}")

refs = ev[REF_COL].fillna("").astype(str).str.strip().tolist()
hyps = ev[GEN_COL].fillna("").astype(str).str.strip().tolist()
N = len(refs)
empty_idx = [i for i, h in enumerate(hyps) if not h]
print(f"n = {N} | empty generations = {len(empty_idx)}"
      + (f"  (rows {empty_idx[:10]})" if empty_idx else ""))

# ---- 3. ROUGE + BLEU (empty hyp -> 0 naturally) ----
import sacrebleu
from rouge_score import rouge_scorer
class _WS:
    def tokenize(self, t): return str(t).split()
sc = rouge_scorer.RougeScorer(["rouge1", "rouge2", "rougeL"],
                              use_stemmer=False, tokenizer=_WS())
per = [sc.score(r, h) for r, h in zip(refs, hyps)]
r1 = [p["rouge1"].fmeasure for p in per]
r2 = [p["rouge2"].fmeasure for p in per]
rl = [p["rougeL"].fmeasure for p in per]
bl = [0.0 if not h else
      sacrebleu.sentence_bleu(h, [r], smooth_method="exp", tokenize="intl").score / 100
      for h, r in zip(hyps, refs)]
cb = sacrebleu.corpus_bleu(hyps, [refs], tokenize="intl")

# ---- 4. BERTScore on NON-EMPTY pairs only; empties get 0.0 ----
from bert_score import score as bert_score_fn
ok = [i for i in range(N) if hyps[i] and refs[i]]
bs = [0.0] * N
if ok:
    _, _, F = bert_score_fn([hyps[i] for i in ok], [refs[i] for i in ok],
                            lang="hi", model_type="bert-base-multilingual-cased",
                            batch_size=8, verbose=False)
    for j, i in enumerate(ok):
        bs[i] = float(F[j])
print(f"BERTScore computed on {len(ok)}/{N}; {N-len(ok)} empty scored 0.0")

# ---- 5. aggregate with bootstrap CIs ----
def ci(v, n=1000):
    rng = np.random.default_rng(SEED); a = np.asarray(v, float)
    b = [a[rng.integers(0, len(a), len(a))].mean() for _ in range(n)]
    return [float(np.percentile(b, 2.5)), float(np.percentile(b, 97.5))]

lens = [len(h.split()) for h in hyps]
ntok = [NTOK.get(k, np.nan) for k in ev["row_key"]] if NTOK else [np.nan] * N
hit_cap = (float(np.nanmean([n >= MAX_NEW_TOKENS - 1 for n in ntok]))
           if NTOK else None)

M = {"run": RUN_TAG, "model": MODEL_NAME, "lang": LANG, "task": TASK, "shots": SHOTS,
     "n": N, "n_empty": len(empty_idx),
     "valid_rate": float(1 - len(empty_idx) / N),
     "bleu_sentence": float(np.mean(bl)), "bleu_sentence_ci": ci(bl),
     "rouge1": float(np.mean(r1)), "rouge1_ci": ci(r1),
     "rouge2": float(np.mean(r2)), "rouge2_ci": ci(r2),
     "rougeL": float(np.mean(rl)), "rougeL_ci": ci(rl),
     "bertscore_f1": float(np.mean(bs)), "bertscore_f1_ci": ci(bs),
     "bertscore_f1_nonempty": float(np.mean([bs[i] for i in ok])) if ok else 0.0,
     "brevity_penalty": float(cb.bp), "bleu_corpus": float(cb.score / 100),
     "len_mean": float(np.mean(lens)), "len_median": float(np.median(lens)),
     "len_std": float(np.std(lens)),
     "pct_80_100w": float(np.mean([80 <= l <= 100 for l in lens])),
     "pct_hit_token_cap": hit_cap}

print(f"\n{'='*64}\n{RUN_TAG}   n={N}   valid={M['valid_rate']:.1%}\n{'='*64}")
for k in ("bleu_sentence", "rouge1", "rouge2", "rougeL", "bertscore_f1"):
    lo, hi = M[k + "_ci"]
    print(f"  {k:<15} {M[k]:.4f}   95% CI [{lo:.4f}, {hi:.4f}]")
print(f"  bertscore (non-empty only) {M['bertscore_f1_nonempty']:.4f}")
print(f"  brevity_penalty {M['brevity_penalty']:.4f}")
print(f"  length mean={M['len_mean']:.1f} med={M['len_median']:.0f} "
      f"in 80-100w={M['pct_80_100w']:.1%}")
if hit_cap is not None:
    print(f"  hit {MAX_NEW_TOKENS}-token cap: {hit_cap:.1%}")
else:
    print("  hit-token-cap: n/a (kernel restarted; token counts were in memory)")

pd.DataFrame({"row_key": ev["row_key"], "ref": refs, "hyp": hyps,
              "rouge1": r1, "rouge2": r2, "rougeL": rl, "bleu": bl,
              "bertscore": bs, "len": lens, "ntok": ntok}
             ).to_csv(PERITEM_CSV, index=False, encoding="utf-8-sig")
json.dump(M, open(METRICS_JSON, "w"), indent=2, ensure_ascii=False)
print(f"\nsaved {PERITEM_CSV}\nsaved {METRICS_JSON}")

if empty_idx:
    print(f"\nEmpty generations at test rows {empty_idx} — report valid_rate "
          f"({M['valid_rate']:.1%}) in the paper; these count as 0, not dropped.")


using generations already in memory
n = 201 | empty generations = 0


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

[transformers] BertModel LOAD REPORT from: bert-base-multilingual-cased
Key                                        | Status     |  | 
-------------------------------------------+------------+--+-
cls.seq_relationship.weight                | UNEXPECTED |  | 
cls.predictions.transform.dense.weight     | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED |  | 
cls.seq_relationship.bias                  | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED |  | 
cls.predictions.bias                       | UNEXPECTED |  | 
cls.predictions.transform.dense.bias       | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


BERTScore computed on 201/201; 0 empty scored 0.0

llama31_mai_summary_zeroshot   n=201   valid=100.0%
  bleu_sentence   0.0060   95% CI [0.0051, 0.0069]
  rouge1          0.0588   95% CI [0.0523, 0.0652]
  rouge2          0.0017   95% CI [0.0010, 0.0026]
  rougeL          0.0460   95% CI [0.0413, 0.0508]
  bertscore_f1    0.6314   95% CI [0.6272, 0.6361]
  bertscore (non-empty only) 0.6314
  brevity_penalty 0.4325
  length mean=41.3 med=44 in 80-100w=4.5%
  hit 200-token cap: 30.3%

saved per_item_llama31_mai_summary_zeroshot.csv
saved metrics_llama31_mai_summary_zeroshot.json


In [12]:
# ============================================================
# CELL 10 — protocol manifest (Section V-B) + failure-mode sample + cleanup
# ============================================================
json.dump({"run":RUN_TAG, "model_name":MODEL_NAME, "arch":ARCH, "chat_format":FMT,
           "language":LANG, "task":TASK, "eval_set":TEST_CSV, "n_eval":EXPECT_N,
           "split_seed":SEED, "reference_column":REF_COL, "shots":SHOTS, "exemplar":None,
           "prompt_sha1":PROMPT_SHA1, "affect_clause_in_prompt":True,
           "decoding":{"num_beams":NUM_BEAMS, "length_penalty":LENGTH_PENALTY,
                       "no_repeat_ngram_size":NO_REPEAT_NGRAM, "do_sample":DO_SAMPLE,
                       "max_new_tokens":MAX_NEW_TOKENS, "max_input_tokens":MAX_INPUT_TOKENS},
           "precision":"float32" if ARCH=="seq2seq" else "bfloat16",
           "attn_implementation":None if ARCH=="seq2seq" else "eager",
           "postprocessing":"marker + label cleanup only; NO length cap",
           "empty_output_policy":"scored as 0, never dropped",
           "metrics":{"rouge":"word-level whitespace, per-instance F1",
                      "bleu":"sentence-level, smooth=exp, tokenize=intl, /100",
                      "bertscore":"bert-base-multilingual-cased, lang=hi",
                      "ci":"1000-sample bootstrap, 95%"}},
          open(MANIFEST_JSON, "w"), indent=2, ensure_ascii=False)
print("saved", MANIFEST_JSON)

print("\n--- 15 outputs for failure-mode notes (Hindi? repetition? echo? truncated?) ---")
for i in df.index[:15]:
    h = df.at[i, GEN_COL]; k = df.at[i, "row_key"]
    print(f"[{len(h.split()):>3}w {NTOK.get(k,0):>3}tok] {h[:110] if h else '<EMPTY>'}")

del model, tokenizer; gc.collect(); torch.cuda.empty_cache()
print("\nGPU released.")


saved manifest_llama31_mai_summary_zeroshot.json

--- 15 outputs for failure-mode notes (Hindi? repetition? echo? truncated?) ---
[ 38w 123tok] अछी। अर्थार्जन के साथ-साथ हमरा सामाजिक, सांस्कृतिक, राजनीतिक आ शैक्षिक क्षेत्र मे भी प्रगति करय के प्रयास करय 
[ 76w 200tok] अर्थ: प्रवीन नारायन चौधारी जी के कार्तून मे मिथीली भासा का फिल्म छोड़कर अश्लिल और फूहड़ भोज पुरी फिलिम की ओर स
[ 62w 200tok] जातिसभक नाम लिखल गेलैक, जिनमे सँ वैश्य, क्षत्रिय, शूद्र आ ब्राम्हण छथि। ५ वैश्यक समुदाय के बारे मे विशेष रूपसँ
[ 18w  49tok] ी बारेमे सूचित केलनि। अर्थात, राजेंश झा आ संजाय झा के बीच के मिलन का क्या परिणाम हुआ?
[ 61w 166tok] आँहा याद तड़पैत रहै, बिछुड़क पीड़ा बहुत सतैत रहै। रात कटै सजनी, मन सांझे विरह गीत कै गाबै रहै। आंखिक सोजा सँ ह
[ 75w 200tok] ेल के खेत बेचि के देला छल। किन्तु बेटि सबके विद्यार्थी जीवन मे प्रगति नहि होइत गेल। छोटी छोटि काम मे भूलभूल कर
[ 10w  29tok] संचालन कार्यक्रम समिति केर सदस्य प्रेम कुमार सिंह ने केलक।
[ 44w 143tok] ानकित केला गेल। ४. संयुक्त अरब अमीरात (युएई) केर राजधानी अ


GPU released.
